___
<h1> Machine Learning </h1>
<h2> M. Sc. in Electrical and Computer Engineering </h2>
<h3> Instituto Superior de Engenharia / Universidade do Algarve </h3>

[MEEC](https://ise.ualg.pt/en/curso/1477) / [ISE](https://ise.ualg.pt) / [UAlg](https://www.ualg.pt)

Pedro J. S. Cardoso (pcardoso@ualg.pt)
___

# Cross validation

Cross validation reuses a dataset as both training and test data, one fold at a time. It is useful when data are limited.

If the score will be used to choose a model or a hyperparameter, lock a test set first. Cross-validation then runs only on the remaining training data. The locked test set is scored once, after that choice.


|# train|Fold 1 |Fold 2 |Fold 3 |Fold 4 |Fold 5|
|-------|-------|-------|-------|-------|------|
|1      |__Validation__   |Train  |Train  |Train  |Train |
|2      |Train  |__Validation__   |Train  |Train  |Train |
|3      |Train  |Train  |__Validation__   |Train  |Train |
|4      |Train  |Train  |Train  |__Validation__   |Train |
|5      |Train  |Train  |Train  |Train  |__Validation__  |

The process is as follows:
1. Lock a test set if you will choose a model or a hyperparameter from the scores
2. Split the remaining data into k parts (folds)
3. Train the model on k-1 parts and score the held-out fold
4. Repeat k times, each time using a different fold for validation
5. Compute the average performance, and optionally the standard deviation
6. Refit on the whole training split, not on the locked test set, then score the test set once

There are some drawbacks to this method, such as:
- It is computationally expensive, as it requires training the model k times
- It does not by itself return the final predictor: each fold trains on a different subset
- It can be sensitive to the way the data is split
- It can be sensitive to the choice of k


Using scikit-learn, cross-validation can be done using the `cross_val_score` function. Let us see an example using the iris dataset and a logistic regression model.

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

Load the iris dataset and define a logistic regression model.

The test set is locked before the first cross-validation. Those scores use only the training split. A later section uses the full Iris set, in its original order, to show how splitters behave.

In [ ]:
iris = load_iris()

X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.25,
    stratify=iris.target,
    random_state=1,
)

log_reg = LogisticRegression(
                        random_state=1,
                        C=200,
                        solver='lbfgs',
                        max_iter=10000)

Cross-validation now estimates accuracy on the training split. The test set is not used here.

In [ ]:
scores = cross_val_score(estimator=log_reg, # model
                         X=X_train, y=y_train,
                         cv=3,       #number of folds - default 5-fold cross validation (see alternatives in documentation)
                         n_jobs=-1,  # use all CPU
                         verbose=3,   # verbose level
                        )

The computed scores, which are the accuracy of the model in each fold, are stored in the variable `scores`.

In [ ]:
scores

Average CV score on the training split. This is not yet the locked test accuracy.

In [ ]:
scores.mean()

And the standard deviation

In [ ]:
scores.std()

The spread of those fold scores shows how sensitive the model is to which training rows are held out inside the training split. It is not a substitute for the locked test score.

The main disadvantage of cross-validation is increased computational cost. As we are now training k models instead of a single model, cross-validation will be roughly $k$ times slower than doing a single split of the data.

## One score on the locked test set

Cross-validation trained five models. The cheaper check is a single fit on `X_train`.

The next cell reports both numbers:

- **mean CV accuracy** — average of the five validation folds
- **held-out test accuracy** — one score on `X_test`, which none of those fits used


In [ ]:
log_reg.fit(X_train, y_train)
print(f"mean CV accuracy:       {scores.mean():.3f}")
print(f"held-out test accuracy: {log_reg.score(X_test, y_test):.3f}")


# Stratified k-Fold Cross-Validation and Other Strategies

The cells in this section use the full Iris set in its original order. They show how a splitter behaves. They are not a second look at the locked test set.

## k-fold's

Splitting the dataset into k folds by starting with the first one-k-th part of the data, as described in the previous section, might not always be a good idea.

How would a 3 fold CV work over a dataset with the data ordered as in the iris dataset!?

In [ ]:
iris.target

The first scores stayed high because `cross_val_score`, for classification, uses **stratified k-fold** by default: each fold keeps the class proportions of the data it is given.

Plain `KFold` does not. On the full Iris set, in its original class order, that matters.

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html

In [ ]:
from sklearn.model_selection import KFold
kfold = KFold(n_splits=3, 
              shuffle=False)

cross_val_score(estimator=log_reg, 
                X=iris.data, 
                y=iris.target, 
                cv=kfold #  int, cross-validation generator or an iterable, optional
               )

0! Why!!????? Can you guess?

with a 5-fold we have some improvements, but...

In [ ]:
kfold = KFold(n_splits=5, 
              shuffle=False)

cross_val_score(estimator=log_reg, 
                X=iris.data, 
                y=iris.target, 
                cv=kfold #  int, cross-validation generator or an iterable, optional
               )

Kfold class allows us to shuffle the data
(https://scikit-learn.org/stable/modules/classes.html#module-sklearn.model_selection)

In [ ]:
kfold = KFold(n_splits=3, 
              shuffle=True, 
              random_state=0)

cross_val_score(estimator=log_reg, 
                X=iris.data, 
                y=iris.target, 
                cv=kfold 
               )

## Stratified k-fold

This is the same splitter that `cross_val_score` uses by default for classification. The call below runs on the full Iris set, in its original order, not on the locked training split.

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html#sklearn.model_selection.StratifiedKFold

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold()

score = cross_val_score(estimator=log_reg, 
                X=iris.data, 
                y=iris.target, 
                cv=skf #  int, cross-validation generator or an iterable, optional
               )
score

## Leave-one-out CV

Leave-one-out cross-validation:  is as $k$-fold cross-validation where each fold is a single sample. For each split, you pick a single data point to be the test set. 
        
This can be very time consuming, particularly for large datasets, but sometimes provides better estimates on small datasets

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html#sklearn.model_selection.LeaveOneOut


In [ ]:
from sklearn.model_selection import LeaveOneOut
loo = LeaveOneOut()

score = cross_val_score(estimator=log_reg, 
                X=iris.data, 
                y=iris.target, 
                cv=loo, #  int, cross-validation generator or an iterable, optional
                verbose=True,
                        n_jobs=-1
               )
score

In [ ]:
len(score)

In [ ]:
score.mean()

In [ ]:
score.std()

In [ ]:
plt.plot(score)
plt.show()

## Shuffle-split cross-validation

In shuffle-split cross-validation, each split samples `train_size` many points for the training set and `test_size` many (disjoint) point for the test set.
This splitting is repeated `n_iter` times. 

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ShuffleSplit.html#sklearn.model_selection.ShuffleSplit

In [ ]:
from sklearn.model_selection import ShuffleSplit
ss = ShuffleSplit(n_splits=10, 
                  train_size=.75, 
                  #     test_size wil be complement of the train_size
                  random_state=1)

score = cross_val_score(estimator=log_reg, 
                X=iris.data, 
                y=iris.target,  
                cv=ss)
score

In [ ]:
score.mean()

## Stratified Shuffle-split cross-validation
There is also a stratified variant of ShuffleSplit, aptly named StratifiedShuffleSplit, which can provide more reliable results for classification tasks.

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedShuffleSplit.html#sklearn.model_selection.StratifiedShuffleSplit

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(
    n_splits=10, 
    train_size=.75, 
#     test_size wil be complement of the train_size
    random_state=1)

score = cross_val_score(estimator=log_reg, 
                X=iris.data, 
                y=iris.target, 
                cv=sss)
score

In [ ]:
score.mean()

## Hyperparameter search without a full grid

A grid with *k*-fold cross-validation fits one model for every combination and every fold. The strategies below use fewer full trainings. On Iris the gain is the number of fits, not the clock time.

`C` is chosen by cross-validation on the training split from the start of the notebook. The test set is scored only after the search.

https://scikit-learn.org/stable/modules/grid_search.html


In [ ]:
print(f"train: {len(y_train)}   test: {len(y_test)} (locked)")


### Random search

Draw a fixed budget of values instead of every point on a grid. `n_iter` is the number of trainings (times the number of CV folds).

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html


In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000),
    param_distributions={"C": loguniform(1e-2, 1e2)},
    n_iter=15,
    cv=3,
    random_state=1,
    n_jobs=-1,
    verbose=3
)
random_search.fit(X_train, y_train)
print(f"best parameters:        {random_search.best_params_}")
print(f"mean CV accuracy:       {random_search.best_score_:.3f}")
print(f"held-out test accuracy: {random_search.score(X_test, y_test):.3f}")


### Successive halving

Many candidates start on a small sample. At each step the best fraction is kept and the sample grows. Weak values of `C` are dropped before a full training.

Arguments in the next cell:

- `LogisticRegression(max_iter=1000)` — the model being tuned. `max_iter` is only the solver limit, not a value being searched.
- `param_distributions={"C": loguniform(1e-2, 1e2)}` — the only hyperparameter in the search. `C` is drawn between \(10^{-2}\) and \(10^{2}\) on a log scale.
- `factor=2` — each round keeps the best half of the candidates (`1/factor`) and doubles the resource. By default that resource is the number of training rows (`n_samples`).
- `cv=3` — 3-fold cross-validation on `X_train` only. The locked test set is not used to choose `C`.
- `random_state=1` — fixes the random draws, so the same candidates appear again.
- `n_jobs=-1` — use every CPU.
- `verbose=3` — print each fit as it runs.

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.HalvingRandomSearchCV.html


In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

halving = HalvingRandomSearchCV(
    LogisticRegression(max_iter=1000),
    param_distributions={"C": loguniform(1e-2, 1e2)},
    factor=2,
    cv=3,
    random_state=1,
    n_jobs=-1,
    verbose=3
)
halving.fit(X_train, y_train)
print(f"best parameters:        {halving.best_params_}")
print(f"mean CV accuracy:       {halving.best_score_:.3f}")
print(f"held-out test accuracy: {halving.score(X_test, y_test):.3f}")


### Bayesian optimization (Optuna)

The next trial depends on the scores already seen. TPE is Optuna's usual sampler. `n_trials` is the budget.

Parameters in the next cell:

- `suggest_float("C", 1e-2, 1e2, log=True)` — Optuna draws `C` between $10^{-2}$ and $10^{2}$. `log=True` spreads the draws evenly on a log scale, so values near `0.01` and near `100` are both easy to reach.
- `LogisticRegression(C=C, max_iter=1000)` — one model per trial. `max_iter` is only the solver limit, not a hyperparameter being searched.
- `cross_val_score(..., cv=3)` — 3-fold cross-validation on `X_train` only. The locked test set is not used here.
- `return scores.mean()` — the number Optuna tries to improve. One scalar per trial.
- `create_study(direction="maximize")` — higher mean accuracy is better. Use `"minimize"` for a loss.
- `TPESampler(seed=1)` — Tree-structured Parzen Estimator. The seed makes the sequence of trials repeatable.
- `optimize(..., n_trials=15)` — stop after 15 complete trials.

https://optuna.readthedocs.io/


In [ ]:
#!pip install optuna
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    C = trial.suggest_float("C", 1e-2, 1e2, log=True)
    model = LogisticRegression(C=C, max_iter=1000)
    scores = cross_val_score(model, X_train, y_train, cv=3)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=1),
)
study.optimize(objective, n_trials=15)

best_C = study.best_params["C"]
final_model = LogisticRegression(C=best_C, max_iter=1000).fit(X_train, y_train)
print(f"best parameters:        {study.best_params}")
print(f"mean CV accuracy:       {study.best_value:.3f}")
print(f"held-out test accuracy: {final_model.score(X_test, y_test):.3f}")
